# Gemma 3 1B — QLoRA sur Colab T4 (un seul epoch)

Ce notebook ajoute une piste expérimentale sans modifier le notebook starter :

- **Modèle d'inférence :** `google/gemma-3-1b-it` (1B, open weights), chargé en NF4 4 bits.
- **Entraînement :** une époque sur les **15 000 exemples officiels uniquement** ; aucun jeu synthétique, aucun split de validation interne.
- **LoRA :** rang 8 sur les projections d'attention et du MLP ; petits batches et gradient checkpointing pour la T4 15 Go.
- **Inférence :** greedy, extraction JSON et complétion des champs du schéma sans inférer les valeurs manquantes.
- **Sortie :** `submission.zip`, contenant uniquement `predictions.json` à la racine.

Le DEV reste sans labels et sert uniquement à produire les prédictions à soumettre. Ce notebook ne calcule **aucun score local** : la seule mesure de résultat est Codabench. Cette configuration est un candidat à tester, pas une garantie de battre le leaderboard. Si Codabench stagne, ne relance pas une deuxième époque sur les mêmes données.


## 1. Installation

Dans Colab, choisir **Runtime → Change runtime type → T4 GPU**. Accepter auparavant la licence Gemma sur Hugging Face et enregistrer un secret Colab nommé `HF_TOKEN`.


In [ ]:
import subprocess
import sys

packages = [
    "transformers>=4.51,<5",
    "peft>=0.15,<1",
    "accelerate>=1.6,<2",
    "bitsandbytes>=0.45",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dépendances QLoRA installées.")


## 2. Imports, accès au modèle et reproductibilité

Le token Hugging Face est lu depuis le Secret Colab `HF_TOKEN` ou la variable d'environnement du même nom. Il n'est jamais affiché.


In [ ]:
import json
import os
import random
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 2026
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

hf_token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    raise RuntimeError("Ajoute le secret HF_TOKEN dans Colab après avoir accepté la licence Gemma.")
login(token=hf_token, add_to_git_credential=False)
del hf_token

if not torch.cuda.is_available():
    raise RuntimeError("Ce notebook est prévu pour une GPU Colab T4, pas pour le CPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM totale: %.1f Go" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))


## 3. Données officielles

On télécharge `train.jsonl`, `dev.jsonl` et `tools.json` depuis le dépôt officiel. Le notebook exige les 15 000 lignes train et vérifie que le DEV n'a pas de labels ; il ne lit ni ne crée de labels DEV/test.


In [ ]:
DATA_DIR = Path("/content/airf_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_BASE = "https://raw.githubusercontent.com/aims-ai-research-foundations/airf-slm-tool-calling-hackathon/main/data"
for filename in ("tools.json", "train.jsonl", "dev.jsonl"):
    urllib.request.urlretrieve(f"{DATA_BASE}/{filename}", DATA_DIR / filename)


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


raw_tools = json.loads((DATA_DIR / "tools.json").read_text(encoding="utf-8"))
TOOL_SCHEMAS = {tool["name"]: tool for tool in raw_tools}
train_rows = read_jsonl(DATA_DIR / "train.jsonl")
dev_rows = read_jsonl(DATA_DIR / "dev.jsonl")
assert len(train_rows) == 15_000, f"Train officiel attendu: 15 000, reçu: {len(train_rows)}"
assert len(dev_rows) == 2_000, f"DEV attendu: 2 000, reçu: {len(dev_rows)}"
assert all(not ({"tool", "parameters", "tool_calls"} & row.keys()) for row in dev_rows), "Le DEV doit rester sans labels."
assert len({row["id"] for row in train_rows}) == len(train_rows)
assert len({row["id"] for row in dev_rows}) == len(dev_rows)
assert set(TOOL_SCHEMAS) == {
    "send_money", "buy_airtime", "buy_data", "check_balance", "buy_electricity_token",
    "report_outage", "get_crop_price", "get_weather", "convert_currency", "set_reminder",
}
for row in train_rows:
    calls = row.get("tool_calls") or [{"tool": row["tool"], "parameters": row["parameters"]}]
    for call in calls:
        schema = TOOL_SCHEMAS[call["tool"]]
        assert call["tool"] in row["tools"], f"Tool non proposé dans {row['id']}"
        assert set(call["parameters"]) == set(schema["parameters"]), f"Schéma incorrect dans {row['id']}"
print(f"{len(train_rows):,} exemples officiels (schémas vérifiés) · {len(dev_rows):,} requêtes DEV sans labels · {len(TOOL_SCHEMAS)} tools")


## 4. Format de la tâche et exemples d'entraînement

Le prompt montre seulement les tools disponibles pour la requête et leurs schémas. Les règles insistent sur les points où une supposition coûte un exact match : `null`, devise non déduite du pays/opérateur, valeurs partagées, ordre des actions, formats des nombres, dates, heures, lieux, noms et tâches de rappel.

Les exemples single sont entraînés comme un objet JSON ; les requêtes à plusieurs actions comme une liste JSON ordonnée. Les labels sont calculés uniquement sur la réponse assistant.


In [ ]:
MODEL_NAME = "google/gemma-3-1b-it"
MAX_SEQUENCE_LENGTH = 1536

RULES = """You convert an English request into tool calls. Use only tools listed for this request.
Output JSON only, with no markdown or explanation. Return one call object for one action;
return a JSON array of call objects in the order requested for multiple actions. Every call
has exactly two keys: "tool" (the tool name as a string) and "parameters" (an object). Include
every parameter in the selected tool's schema. Use JSON null whenever the request does not explicitly state a value. Never infer a
value from the country, city, name, provider, context, or a default. Do not add parameters.

Extraction rules:
- Keep names, places, crops, providers, networks, and reminder tasks as written, except for
  the required formats below. Drop titles/relationship words from a person's name.
- Use the most specific place for each action. A market name omits the word "market".
- Amounts are JSON numbers without currency (5k = 5000; 1.5 million = 1500000); words for numbers
  are also amounts. For airtime/data, amount is money spent, not a bundle size. Keep phone/meter numbers as strings, remove spaces
  and hyphens, and keep a leading plus sign.
- For send_money and buy_electricity_token, currency is an ISO code only when the currency is
  written with the amount; otherwise currency is null. Never infer it from location or provider.
  Currency words/symbols (case-insensitive) map as follows: naira/₦/N = NGN; Ghana cedi/GH₵/GHC = GHS;
  Kenyan shilling/KSh/bob = KES; Ugandan shilling/USh = UGX; Tanzanian shilling/TSh = TZS;
  Rwandan franc/Frw = RWF; rand/R = ZAR; Zambian kwacha/K = ZMW; birr = ETB;
  dollar/$ = USD; euro/€ = EUR; pound/£ = GBP. For convert_currency, from_currency is the
  money owned and to_currency the requested money; both must be named. A rate request may
  have a null amount.
- For reminders, keep the task wording but remove the reminder phrase and date/time. Dates use
  today, tomorrow, lowercase weekday, or D Month; times use 24-hour HH:MM. Morning alone is
  not a time. Keep actions in request order; propagate a shared value only to actions it applies to.
- Greetings and background information are not parameter values.
"""


def build_messages(row):
    offered = []
    for name in row["tools"]:
        schema = TOOL_SCHEMAS[name]
        params = ", ".join(f"{key}: {value_type}" for key, value_type in schema["parameters"].items())
        offered.append(f"- {name}: {schema['description']} Parameters: {params}")
    system = RULES + "\nAvailable tools for this request:\n" + "\n".join(offered)
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": row["request"]},
    ]


def target_answer(row):
    if "tool_calls" in row:
        calls = row["tool_calls"]
    else:
        calls = [{"tool": row["tool"], "parameters": row["parameters"]}]
    if not calls:
        raise ValueError(f"Réponse train vide: {row.get('id')}")
    return calls[0] if len(calls) == 1 else calls


def target_text(row):
    return json.dumps(target_answer(row), ensure_ascii=False, separators=(",", ":"))

# Inspect formatting only; this does not create or score a held-out split.
example = train_rows[0]
print("INPUT MESSAGES:\n", json.dumps(build_messages(example), ensure_ascii=False, indent=2))
print("\nTARGET JSON:\n", target_text(example))


## 5. Tokenisation avec loss uniquement sur les réponses

Aucun échantillon train n'est réservé comme validation : les 15 000 exemples officiels sont utilisés une fois. Si un exemple dépasse `MAX_SEQUENCE_LENGTH`, l'exécution s'arrête au lieu de tronquer silencieusement la requête ou la réponse.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


class ToolCallDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.items = []
        longest = 0
        for row in rows:
            messages = build_messages(row)
            prefix_ids = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True
            )
            full_ids = tokenizer.apply_chat_template(
                messages + [{"role": "assistant", "content": target_text(row)}],
                tokenize=True,
                add_generation_prompt=False,
            )
            if full_ids[:len(prefix_ids)] != prefix_ids:
                raise RuntimeError("Le chat template ne forme pas un préfixe assistant compatible.")
            if len(full_ids) > MAX_SEQUENCE_LENGTH:
                raise ValueError(
                    f"{row['id']} dépasse MAX_SEQUENCE_LENGTH ({len(full_ids)} > {MAX_SEQUENCE_LENGTH}); "
                    "augmente la limite, ne tronque pas les labels."
                )
            labels = [-100] * len(prefix_ids) + full_ids[len(prefix_ids):]
            if not any(label != -100 for label in labels):
                raise ValueError(f"Aucun token cible pour {row['id']}")
            self.items.append({"input_ids": full_ids, "labels": labels})
            longest = max(longest, len(full_ids))
        print(f"{len(self.items):,} séquences encodées · longueur maximale: {longest} tokens")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


train_dataset = ToolCallDataset(train_rows)


## 6. Chargement QLoRA 4 bits et LoRA léger

T4 utilise fp16 pour le calcul 4 bits (pas bf16). Si le chargement ou le premier batch manque de mémoire, garde une époque et les 15k exemples, mais baisse le batch par appareil de 2 à 1 ; ne passe pas à `all-linear`.


In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    attn_implementation="eager",
    low_cpu_mem_usage=True,
    token=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 7. Une seule époque — aucun score de validation interne

Le batch effectif est 16 (`2 × 8`). Il n'y a ni `eval_dataset`, ni appel à `trainer.evaluate()`, ni sélection du meilleur checkpoint par score train/dev interne.


In [ ]:
def collate_answer_only(features):
    pad_id = tokenizer.pad_token_id
    width = max(len(feature["input_ids"]) for feature in features)
    width = ((width + 7) // 8) * 8
    input_ids, attention_masks, labels = [], [], []
    for feature in features:
        ids = feature["input_ids"]
        target = feature["labels"]
        pad = width - len(ids)
        input_ids.append(ids + [pad_id] * pad)
        attention_masks.append([1] * len(ids) + [0] * pad)
        labels.append(target + [-100] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

OUTPUT_DIR = Path("/content/gemma3-1b-toolcall-qlora")
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    seed=SEED,
    data_seed=SEED,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_answer_only,
)
start = time.time()
trainer.train()
trainer.save_model(str(OUTPUT_DIR))  # adapter, not a merged 1B copy
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Entraînement terminé en {(time.time() - start) / 3600:.2f} h; adapter sauvegardé dans {OUTPUT_DIR}")


## 8. Décodage greedy, validation du schéma et ZIP

Le DEV est utilisé ici uniquement pour l'inférence. Le post-traitement conserve l'ordre des appels, rejette un tool qui n'était pas proposé et ajoute les paramètres manquants avec `null`. Il ne complète jamais une valeur sémantique à partir du contexte.


In [ ]:
"""Schema-safe helpers for the African SLM Tool Calling submission format.

This module deliberately does not infer missing values from a request.  It only
parses a model's JSON, validates offered tool names, fills required-but-omitted
fields with ``None``, and converts values to the JSON types in ``tools.json``.
"""

from __future__ import annotations

import json
import math
import re
import zipfile
from pathlib import Path
from typing import Any, Mapping, Sequence


_NUMBER_RE = re.compile(r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?$")
_COMMA_GROUPED_RE = re.compile(r"^[+-]?\d{1,3}(?:,\d{3})+(?:\.\d+)?(?:[eE][+-]?\d+)?$")
_SPACE_GROUPED_RE = re.compile(r"^[+-]?\d{1,3}(?: \d{3})+(?:\.\d+)?(?:[eE][+-]?\d+)?$")
_NULL_TEXT = {"", "null", "none", "nil"}


def answer_from_training_row(row: Mapping[str, Any]) -> Any:
    """Return a benchmark answer, collapsing a one-call list to one object."""
    if "tool_calls" in row:
        calls = row["tool_calls"]
    else:
        calls = [{"tool": row["tool"], "parameters": row["parameters"]}]
    if not isinstance(calls, list) or not calls:
        raise ValueError(f"Invalid training answer for {row.get('id', '<unknown>')}")
    return calls[0] if len(calls) == 1 else calls


def _as_finite_number(value: Any) -> int | float | None:
    """Convert JSON-ish numeric output to a JSON number; do not parse shorthand."""
    if isinstance(value, bool) or value is None:
        return None
    if isinstance(value, int):
        return value
    if isinstance(value, float):
        if not math.isfinite(value):
            return None
        return int(value) if value.is_integer() else value
    if not isinstance(value, str):
        return None

    text = value.strip()
    if not text or text.lower() in _NULL_TEXT:
        return None
    if _COMMA_GROUPED_RE.fullmatch(text):
        text = text.replace(",", "")
    elif _SPACE_GROUPED_RE.fullmatch(text):
        text = text.replace(" ", "")
    elif not _NUMBER_RE.fullmatch(text):
        return None

    if "." not in text and "e" not in text.lower():
        try:
            return int(text)
        except ValueError:
            return None
    try:
        number = float(text)
    except ValueError:
        return None
    if not math.isfinite(number):
        return None
    if number.is_integer():
        return int(number)
    return number


def _as_string_or_null(value: Any) -> str | None:
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        return None if value.lower() in _NULL_TEXT else value
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, int):
        # A string field (notably phone/meter numbers) must remain a JSON string.
        return str(value)
    if isinstance(value, float) and math.isfinite(value):
        return str(int(value)) if value.is_integer() else str(value)
    return None


def parse_model_json(text: str) -> Any | None:
    """Find the first JSON object/array in a model reply, including fenced output."""
    if not isinstance(text, str):
        return None
    decoder = json.JSONDecoder()
    starts = sorted(pos for pos in (text.find("{"), text.find("[")) if pos >= 0)
    for start in starts:
        try:
            value, _end = decoder.raw_decode(text[start:])
        except json.JSONDecodeError:
            continue
        if isinstance(value, (dict, list)):
            return value
    return None


def fallback_answer(available_tools: Sequence[str], schemas: Mapping[str, Any]) -> dict[str, Any]:
    """Return a well-formed last-resort answer when decoding/validation fails.

    This does not attempt to guess values.  It uses the first offered tool with
    null parameters so the submission remains schema-shaped rather than being
    rejected for malformed output.  The notebook reports how often this occurs.
    """
    if not available_tools:
        return {"tool": "", "parameters": {}}
    name = available_tools[0]
    schema = schemas.get(name, {})
    parameters = schema.get("parameters", {}) if isinstance(schema, Mapping) else {}
    return {"tool": name, "parameters": {key: None for key in parameters}}


def _coerce_call(
    call: Any,
    available_tools: Sequence[str],
    schemas: Mapping[str, Any],
) -> dict[str, Any] | None:
    if not isinstance(call, Mapping):
        return None

    tool_name = call.get("tool", call.get("name"))
    if not isinstance(tool_name, str) or tool_name not in available_tools:
        return None
    schema = schemas.get(tool_name)
    if not isinstance(schema, Mapping):
        return None
    parameter_schema = schema.get("parameters")
    if not isinstance(parameter_schema, Mapping):
        return None

    raw_parameters = call.get("parameters", call.get("arguments", {}))
    if not isinstance(raw_parameters, Mapping):
        raw_parameters = {}

    parameters: dict[str, Any] = {}
    for key, value_type in parameter_schema.items():
        value = raw_parameters.get(key)
        if value_type == "number":
            parameters[key] = _as_finite_number(value)
        elif value_type == "string":
            parameters[key] = _as_string_or_null(value)
        else:
            # Unknown schema types are not silently passed through.
            return None
    return {"tool": tool_name, "parameters": parameters}


def coerce_prediction(
    prediction: Any,
    available_tools: Sequence[str],
    schemas: Mapping[str, Any],
) -> tuple[Any, bool]:
    """Validate and schema-complete an answer; return (answer, used_fallback)."""
    value = parse_model_json(prediction) if isinstance(prediction, str) else prediction
    if isinstance(value, Mapping) and "tool_calls" in value:
        value = value["tool_calls"]

    calls = value if isinstance(value, list) else [value]
    if not calls:
        return fallback_answer(available_tools, schemas), True

    normalized: list[dict[str, Any]] = []
    for call in calls:
        result = _coerce_call(call, available_tools, schemas)
        if result is None:
            return fallback_answer(available_tools, schemas), True
        normalized.append(result)

    # The benchmark treats a one-item list and an object as equivalent; emit the
    # canonical object for one action and preserve order for genuine multi-calls.
    return (normalized[0] if len(normalized) == 1 else normalized), False


def validate_prediction_map(
    rows: Sequence[Mapping[str, Any]],
    predictions: Mapping[str, Any],
    schemas: Mapping[str, Any],
) -> None:
    """Raise on missing/extra IDs or malformed calls before creating a ZIP."""
    row_ids = [str(row["id"]) for row in rows]
    if len(row_ids) != len(set(row_ids)):
        raise ValueError("Input data contains duplicate IDs")
    if set(predictions) != set(row_ids):
        missing = sorted(set(row_ids) - set(predictions))[:5]
        extra = sorted(set(predictions) - set(row_ids))[:5]
        raise ValueError(f"Prediction IDs do not match input IDs (missing={missing}, extra={extra})")

    for row in rows:
        answer = predictions[str(row["id"])]
        calls = answer if isinstance(answer, list) else [answer]
        if not calls:
            raise ValueError(f"Empty call list for {row['id']}")
        available = row.get("tools", [])
        for call in calls:
            if not isinstance(call, Mapping):
                raise ValueError(f"Invalid call for {row['id']}")
            name = call.get("tool")
            if name not in available or name not in schemas:
                raise ValueError(f"Tool {name!r} is not offered for {row['id']}")
            expected_keys = set(schemas[name]["parameters"])
            params = call.get("parameters")
            if not isinstance(params, Mapping) or set(params) != expected_keys:
                raise ValueError(f"Incomplete parameter schema for {row['id']} / {name}")
            for key, value_type in schemas[name]["parameters"].items():
                value = params[key]
                if value is not None and value_type == "number" and (
                    isinstance(value, bool)
                    or not isinstance(value, (int, float))
                    or (isinstance(value, float) and not math.isfinite(value))
                ):
                    raise ValueError(f"Non-numeric {key!r} for {row['id']}")
                if value is not None and value_type == "string" and not isinstance(value, str):
                    raise ValueError(f"Non-string {key!r} for {row['id']}")


def write_submission_zip(
    rows: Sequence[Mapping[str, Any]],
    predictions: Mapping[str, Any],
    schemas: Mapping[str, Any],
    zip_path: str | Path = "submission.zip",
    json_path: str | Path = "predictions.json",
) -> tuple[Path, Path]:
    """Write the exact Codabench format: one root-level predictions.json in ZIP."""
    validate_prediction_map(rows, predictions, schemas)
    ordered = {str(row["id"]): predictions[str(row["id"])] for row in rows}
    json_file = Path(json_path)
    json_file.write_text(
        json.dumps(ordered, ensure_ascii=False, separators=(",", ":"), allow_nan=False),
        encoding="utf-8",
    )

    zip_file = Path(zip_path)
    with zipfile.ZipFile(zip_file, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(json_file, arcname="predictions.json")
    return json_file, zip_file


In [ ]:
tokenizer.padding_side = "left"
model.config.use_cache = True
model.eval()
torch.cuda.empty_cache()

stop_ids = []
for token_id in (tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<end_of_turn>")):
    if token_id is not None and token_id != tokenizer.unk_token_id and token_id not in stop_ids:
        stop_ids.append(token_id)
if not stop_ids:
    stop_ids = [tokenizer.eos_token_id]


def predict_rows(rows, batch_size=2, max_new_tokens=384):
    predictions = {}
    fallback_count = 0
    # Bucketing by prompt length reduces padding without changing result order.
    ordered = sorted(rows, key=lambda row: len(row["request"]))
    for start_idx in range(0, len(ordered), batch_size):
        batch = ordered[start_idx:start_idx + batch_size]
        prompts = [
            tokenizer.apply_chat_template(build_messages(row), tokenize=False, add_generation_prompt=True)
            for row in batch
        ]
        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        )
        if encoded["input_ids"].shape[1] > MAX_SEQUENCE_LENGTH:
            raise ValueError("Un prompt DEV dépasse la limite de contexte; augmente MAX_SEQUENCE_LENGTH.")
        encoded = {key: value.to(model.device) for key, value in encoded.items()}
        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                eos_token_id=stop_ids,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
            )
        continuation = generated[:, encoded["input_ids"].shape[1]:]
        texts = tokenizer.batch_decode(continuation, skip_special_tokens=True)
        for row, text in zip(batch, texts):
            answer, used_fallback = coerce_prediction(text, row["tools"], TOOL_SCHEMAS)
            predictions[str(row["id"])] = answer
            fallback_count += int(used_fallback)
    return predictions, fallback_count

predictions, fallback_count = predict_rows(dev_rows, batch_size=2, max_new_tokens=384)
# Restore original input order for a stable, easy-to-audit submission.
predictions = {str(row["id"]): predictions[str(row["id"])] for row in dev_rows}
json_path, zip_path = write_submission_zip(
    dev_rows,
    predictions,
    TOOL_SCHEMAS,
    zip_path="submission.zip",
    json_path="predictions.json",
)
assert len(predictions) == len(dev_rows)
with zipfile.ZipFile(zip_path) as archive:
    assert archive.namelist() == ["predictions.json"]
print(f"{len(predictions):,} prédictions · {fallback_count} sortie(s) de secours · ZIP prêt: {zip_path}")


## 9. Récupérer et soumettre

Télécharge `submission.zip` puis soumets-le sur Codabench. **N'interprète pas la loss d'entraînement comme un score et ne relance pas une seconde époque si le DEV stagne.** La décision suivante se prend uniquement avec l'Exact-Match Accuracy Codabench et le quota de soumissions disponible.


In [ ]:
try:
    from google.colab import files
    files.download("submission.zip")
except ImportError:
    print("Fichier prêt dans:", Path("submission.zip").resolve())
